In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

project_path = "/content/drive/My Drive/meme-analysis-ds"

print(f"Changing directory to: {project_path}")
os.chdir(project_path)

print("\nFiles in project folder:")
!ls

In [ ]:
!pip install --upgrade transformers
!pip install tesseract
!pip install gradio torch pytesseract opencv-python-headless pillow datasets

!apt-get install tesseract-ocr

In [ ]:
import gradio as gr
import torch
from transformers import pipeline, ViTImageProcessor, ViTForImageClassification
import pytesseract
import cv2
import numpy as np
from PIL import Image
import warnings
import os
import torch.nn.functional as F

warnings.filterwarnings("ignore")
print("All libraries installed and imported successfully.")

FINE_TUNED_MODEL_PATH = "./my-awesome-meme-classifier"

if not os.path.exists(FINE_TUNED_MODEL_PATH):
    print("="*50)
    print(f"ERROR: Model directory not found at '{FINE_TUNED_MODEL_PATH}'")
    raise SystemExit("Fine-tuned model not found.")
else:
    print(f"Found fine-tuned model at: {FINE_TUNED_MODEL_PATH}")

print("Loading AI models...")

#Fine-Tuned Model
processor = ViTImageProcessor.from_pretrained(FINE_TUNED_MODEL_PATH)
classification_model = ViTForImageClassification.from_pretrained(FINE_TUNED_MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classification_model.to(device)
print(f"Loaded fine-tuned classification model onto device: {device}")

# Other Pre-trained Models
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)
print("Loaded sentiment model.")

object_detection_pipeline = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
    device=0 if torch.cuda.is_available() else -1
)
print("Loaded object detection model.")

print("All models loaded successfully.")


#OCR function
def extract_text_from_image(pil_image):
    try:
        image_np = np.array(pil_image.convert('RGB'))
        image_cv = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
        width = int(gray.shape[1] * 2)
        height = int(gray.shape[0] * 2)
        dim = (width, height)
        gray = cv2.resize(gray, dim, interpolation=cv2.INTER_LINEAR)
        (T, thresh_img) = cv2.threshold(gray, 0, 255,
                                       cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        thresh_img = cv2.medianBlur(thresh_img, 3)
        text = pytesseract.image_to_string(thresh_img)

        if not text.strip():
            return "No text detected."
        return text.strip()

    except Exception as e:
        print(f"OCR Error: {e}")
        return "Error: Tesseract OCR failed."

#Sentiment function
def get_sentiment(text):
    if text.startswith("Error") or text == "No text detected." or not text:
        return "N/A (No text to analyze)", "N/A"
    try:
        result = sentiment_pipeline(text)[0]
        label = result['label']
        score = result['score']
        # Return label and formatted string separately
        return label, f"{label} ({score * 100:.1f}%)"
    except Exception as e:
        return "N/A", "Error during sentiment analysis."

#Classification function
def get_meme_classification(pil_image):
    try:
        inputs = processor(images=pil_image, return_tensors="pt")
        pixel_values = inputs.pixel_values.to(device)

        with torch.no_grad():
            outputs = classification_model(pixel_values)

        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        score = probs.max().item()
        predicted_class_idx = probs.argmax(-1).item()
        label = classification_model.config.id2label[predicted_class_idx]

        return label, score

    except Exception as e:
        print(f"Error in Classification: {e}")
        return "Error", 0.0

#Object Detection function
def get_identified_objects(pil_image):
    try:
        results = object_detection_pipeline(pil_image)
        if not results:
            return "No objects detected."

        detected_items = []
        raw_labels = []
        for item in results:
            score = item['score']
            if score > 0.5:
                label = item['label']
                detected_items.append(f"{label} ({score*100:.0f}%)")
                raw_labels.append(label)

        if not detected_items:
            return "No objects detected (above 50% confidence).", []

        return ", ".join(list(set(detected_items))), list(set(raw_labels))

    except Exception as e:
        return "Error during object detection.", []


def calculate_virality_score(classification_label, sentiment_label, ocr_text, object_labels):
    """
    Calculates a simple heuristic "virality score" from 0-100.
    """
    score = 0
    reasons = []

    if classification_label == "very_positive":
        score += 50
        reasons.append("Highly Positive Content (+50)")
    elif classification_label == "positive":
        score += 30
        reasons.append("Positive Content (+30)")
    elif classification_label == "negative":
        score -= 20
        reasons.append("Negative Content (-20)")
    elif classification_label == "neutral":
        score += 10
        reasons.append("Neutral/Relatable Content (+10)")

    if sentiment_label == "POSITIVE":
        score += 20
        reasons.append("Positive Text Sentiment (+20)")
    elif sentiment_label == "NEGATIVE":
        score -= 10
        reasons.append("Negative Text Sentiment (-10)")

    if ocr_text != "No text detected.":
        score += 20
        reasons.append("Text is Present (+20)")

    if "person" in object_labels:
        score += 10
        reasons.append("Features a Person (+10)")

    if score < 0:
        score = 0
    if score > 100:
        score = 100

    if score >= 80:
        prediction = "High Potential"
    elif score >= 50:
        prediction = "Medium Potential"
    else:
        prediction = "Low Potential"

    return f"{prediction} (Score: {score}/100)", "\n".join(reasons)

def analyze_meme(uploaded_image):

    # Run all the AI models
    extracted_text = extract_text_from_image(uploaded_image)
    sentiment_label, sentiment_string = get_sentiment(extracted_text)
    classification, confidence_score = get_meme_classification(uploaded_image)
    objects_string, object_labels_list = get_identified_objects(uploaded_image)

    # Run the new virality calculator
    virality_prediction, virality_reasons = calculate_virality_score(
        classification, sentiment_label, extracted_text, object_labels_list
    )

    # We must return 7 items to match the 7 UI outputs
    return (
        classification,
        f"{confidence_score * 100:.2f}%",
        sentiment_string,
        extracted_text,
        objects_string,
        virality_prediction,
        virality_reasons
    )

#Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Meme Analyser (Fine-Tuned)")
    gr.Markdown("Upload a meme and the AI will analyze it. **Now with Virality Potential!**")

    with gr.Row(variant="panel"):
        # INPUT
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Your Meme")
            submit_button = gr.Button("Analyze Meme", variant="primary")

        # OUTPUT
        with gr.Column(scale=2):

            output_virality = gr.Textbox(label="Virality Potential")
            output_virality_reasons = gr.Textbox(label="Virality Analysis", lines=3)

            gr.Markdown("---")

            output_classification = gr.Textbox(label="Meme Classification (Fine-Tuned)")
            output_confidence = gr.Textbox(label="Classification Confidence")
            output_sentiment = gr.Textbox(label="Sentiment Analysis")
            output_ocr = gr.Textbox(label="Extracted Text (OCR) - v2")
            output_objects = gr.Textbox(label="Identified Objects (with score)")

    submit_button.click(
        fn=analyze_meme,
        inputs=image_input,
        outputs=[
            output_classification,
            output_confidence,
            output_sentiment,
            output_ocr,
            output_objects,
            output_virality,
            output_virality_reasons
        ]
    )

    # Examples
    try:
        gr.Examples(
            examples=[
                "images/images/image_1.jpg",
                "images/images/image_100.jpg",
                "images/images/image_6825.jpg"
            ],
            inputs=image_input,

            outputs=[
                output_classification,
                output_confidence,
                output_sentiment,
                output_ocr,
                output_objects,
                output_virality,
                output_virality_reasons
            ],
            fn=analyze_meme
        )
    except FileNotFoundError:
        print("Could not load examples. Make sure the 'images' folder is present.")


print("Launching Gradio Interface...")
demo.launch(debug=True, share=True)